In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn.functional as fun

from torchvision import datasets, transforms
from matplotlib import pyplot as plt

import import_ipynb
from common import test_model_sgd_1, OneHot # type: ignore
from per_lin_softmax_ce import Per_Lin_Softmax_CE_Autograd, \
                               Per_Lin_Softmax_CE_Backward, \
                               Per_Lin_Softmax_CE_Gradient # type: ignore

from per_lin_relu_lin_softmax_ce import Per_Lin_ReLU_Lin_Softmax_CE_Autograd # type: ignore

In [ ]:
#
# Load the dataset and show the first 10 samples.
#

to_tensor = transforms.ToTensor()
dataset = datasets.MNIST(root='data', train=True, download=True, transform=to_tensor)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    ax.imshow(dataset.data[idx], cmap='gray')
    ax.set_title(f"Label: {dataset.targets[idx].item()}")
    ax.axis('off')

plt.show()

In [ ]:
#
# Train model
#
S=10000

samples_train = dataset.data[:S].float() / 255.0
samples_train = tr.reshape(samples_train, (samples_train.shape[0], -1))

labels_train = dataset.targets[:S]
labels_train = labels_train[:S]

In [ ]:
modelA = Per_Lin_Softmax_CE_Autograd(in_features=28*28, out_features=10)
print(f"Accuracy model 1/1: {100 * test_model_sgd_1(modelA, samples_train, labels_train, epochs=1000, lr=0.1):.2f}%")

modelB = Per_Lin_Softmax_CE_Backward(in_features=28*28, out_features=10)
print(f"Accuracy model 1/2: {100 * test_model_sgd_1(modelB, samples_train, labels_train, epochs=1000, lr=0.1):.2f}%")

modelG = Per_Lin_Softmax_CE_Gradient(in_features=28*28, out_features=10)
print(f"Accuracy model 1/3: {100 * test_model_sgd_1(modelG, samples_train, labels_train, epochs=1000, lr=0.1):.2f}%")

modelNL = Per_Lin_ReLU_Lin_Softmax_CE_Autograd(in_features=28*28, hidden_features=36, out_features=10)
print(f"Accuracy model 2/1: {100 * test_model_sgd_1(modelNL, samples_train, labels_train, epochs=2000, lr=0.1):.2f}%")

In [ ]:
#
# Show the weights as images.
#

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    weights = modelG.linear.weight[idx].reshape(28, 28).detach().numpy()

    lim = float(abs(weights).max())
    rg_cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
        "red_white_green", ["red", "white", "green"]
    )

    ax.imshow(weights)
    ax.images[-1].set_cmap(rg_cmap)
    ax.images[-1].set_clim(-lim, lim)

    ax.set_title(f"Digit {idx}")
    ax.axis('off')

plt.show()

In [ ]:
#
# Show the weights as images.
#

fig, axes = plt.subplots(12, 3, figsize=(12, 48))

for idx, ax in enumerate(axes.flat):
    weights = modelNL.linear1.weight[idx].reshape(28, 28).detach().numpy()

    lim = float(abs(weights).max())
    rg_cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
        "red_white_green", ["red", "white", "green"]
    )

    ax.imshow(weights)
    ax.images[-1].set_cmap(rg_cmap)
    ax.images[-1].set_clim(-lim, lim)

    ax.set_title(f"1Feature {idx}")
    ax.axis('off')

plt.show()

In [ ]:
#
# Show the weights as images.
#

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    weights = modelNL.linear2.weight[idx].reshape(6, 6).detach().numpy()

    for y in range(6):
        for x in range(6):
            ax.text(x, y, str(6 * y + x), ha='center', va='center', color='white', fontsize=8)

    lim = float(abs(weights).max())
    rg_cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
        "red_white_green", ["red", "white", "green"]
    )

    ax.imshow(weights)
    ax.images[-1].set_cmap(rg_cmap)
    ax.images[-1].set_clim(-lim, lim)

    ax.set_title(f"2Feature {idx}")
    ax.axis('off')

plt.show()

In [ ]:
#
# Run more tests and show the confusion matrix.
#

expected_labels = labels_train
actual_labels = modelNL.predict(samples_train)
confusion_matrix = tr.zeros((10, 10), dtype=tr.int32)

for a, e in zip(actual_labels, expected_labels):
    confusion_matrix[e][a] += 1

confusion_matrix = confusion_matrix.log2()

fig, axes = plt.subplots(1, 1, figsize=(12, 12))

axes.imshow(confusion_matrix, cmap='Blues')
axes.set_title('(Log2) Confusion Matrix')
axes.set_xlabel('Predicted Label')
axes.set_ylabel('Actual Label')
axes.set_xticks(range(10))   
axes.set_yticks(range(10))

plt.colorbar(axes.imshow(confusion_matrix, cmap='Blues'), ax=axes)
plt.show()